## 1. 두쫀쿠 프로젝트

In [20]:
from dotenv import load_dotenv
#환경변수 로드(.env파일 생성해서 OPEN_API_KEY,TAVILY_API_KEY,NAVER TREND API홈페이지에서 CLIENT_ID, CLIENT_SECRET키 발급받아 입력해야함 ) 
load_dotenv()

import re
import os, json

from textwrap import dedent
from pprint import pprint

import warnings
warnings.filterwarnings("ignore")

from langchain_community.tools import TavilySearchResults
from langchain_core.tools import tool
from typing import List

# =====================================================================
# [1단계] Agent가 사용할 도구(Tools) 정의하기
# =====================================================================
# Agent는 함수의 '이름'과 'Docstring(함수 설명)'을 읽고 
# 언제 이 도구를 써야 할지 스스로 판단함. 따라서 Docstring을 명확히 적는 것이 매우 중요! 함수설명은 함수 바로아래 """이부분"""

# Tool 정의 
@tool
def search_web(query: str) -> str:
    """Searches the internet for information that does not exist in the database or for the latest information."""
    # Tavily는 AI 에이전트를 위해 만들어진 검색 엔진
    tavily_search = TavilySearchResults(max_results=2)
    docs = tavily_search.invoke(query)
    # 검색 결과를 Agent가 읽기 편한 문자열 형태로 포매팅
    formatted_docs = "\n---\n".join([
        f'<Document href="{doc["url"]}"/>\n{doc["content"]}\n</Document>'
        for doc in docs
        ])

    if len(formatted_docs) > 0:
        return formatted_docs
    
    return "관련 정보를 찾을 수 없습니다."

In [21]:
def analyze_trend(data):
    """Fetches search trend data from Naver DataLab, analyzes it, and generates a concise summary of key insights."""

    #[버그방어]슈냥이츄밥 같은 검색데이터가 없는 경우를 위한 방어 로직, Index error발생으로 인한 추가
    if not data:
        return {
            "demand": "No demand",
            "overall_trend": "데이터 없음",
            "recent_trend": "데이터 없음",
            "average_interest": 0,
            "latest_interest": 0,
            "trend_speed": "none",
            "recommendation": "Not recommended (검색량 없음)"
        }
    
    ratios = [d["ratio"] for d in data]
    periods = [d["period"] for d in data]

    first = ratios[0]
    last = ratios[-1]

    # 전체 추세
    if last > first:
        overall = "상승"
    elif last < first:
        overall = "하락"
    else:
        overall = "유지"

    # 최근 추세 (마지막 3개월)
    recent = ratios[-3:]

    if recent[-1] > recent[0]:
        recent_trend = "최근 상승"
    elif recent[-1] < recent[0]:
        recent_trend = "최근 하락"
    else:
        recent_trend = "최근 유지"

    avg = sum(ratios) / len(ratios)

    #근 한달 검색량 토대로 추천/비추천 
    if last > 70:
        demand = "High demand"
    elif last > 40:
        demand = "Moderate demand"
    else:
        demand = "Low demand"

    # slope
    slope = (last - first) / len(ratios)

    if slope > 3:
        trend_speed = "rapid growth"
    elif slope > 1:
        trend_speed = "growing"
    elif slope > -1:
        trend_speed = "stable"
    else:
        trend_speed = "declining"

    if demand == "High demand" and trend_speed in ["rapid growth", "growing"]:
        recommendation = "Highly recommended"

    elif demand == "Moderate demand" and trend_speed in ["rapid growth", "growing"]:
        recommendation = "Good opportunity"

    elif demand == "High demand" and trend_speed == "stable":
        recommendation = "Possible but competitive"

    elif trend_speed == "declining":
        recommendation = "Not recommended"

    else:
        recommendation = "Uncertain"

    return {
        "demand": demand,
        "overall_trend": overall,
        "recent_trend": recent_trend,
        "average_interest": avg,
        "latest_interest": ratios[-1],
        "trend_speed":trend_speed,
        "recommendation": recommendation
    }

In [22]:
import requests
import json
from langchain_core.tools import tool
from typing import List
from langchain_core.documents import Document

#.env에 키값 잘 정의해놓기
CLIENT_ID = os.getenv("CLIENT_ID")
CLIENT_SECRET = os.getenv("CLIENT_SECRET")

@tool
def naver_trend(keyword: str) -> str:
    """Searches the naver trend for information that does not exist in the database or for the latest information."""
    # 외부 API를 LangChain Tool로 감싸는 전형적인 방법
    url = "https://naveropenapi.apigw.ntruss.com/datalab/v1/search"

    headers = {
        "X-NCP-APIGW-API-KEY-ID": CLIENT_ID,
        "X-NCP-APIGW-API-KEY": CLIENT_SECRET,
        "Content-Type": "application/json"
    }

    body = {
        "startDate": "2025-01-01",
        "endDate": "2026-03-01",
        "timeUnit": "month",
        "keywordGroups": [
            {
                "groupName": keyword,
                "keywords": [keyword]
            }
        ]
    }

    res = requests.post(url, headers=headers, json=body)

    print(res.status_code)
    print(res.text)

    #분석 결과를 LLM이 이해하기 쉬운 텍스트 형식으로 반환
    data = res.json()
    trend_data = data["results"][0]["data"]
    analysis = analyze_trend(trend_data)

    return f"""
Keyword: {keyword}

recommend highly or lowly: {analysis["demand"]}

Overall trend (2025-2026): {analysis["overall_trend"]}
Recent trend (last 3 months): {analysis["recent_trend"]}

Latest interest score: {analysis["latest_interest"]}
Average interest score: {analysis["average_interest"]:.2f}

trend speed(2025-2026): {analysis["trend_speed"]}
Startup recommendation: {analysis["recommendation"]}
"""








In [23]:
# =====================================================================
# [2단계] 로컬 데이터(RAG) 준비 및 Vector Store 구축하기
# =====================================================================
from langchain.document_loaders import TextLoader

# 내가 프로토타입을 위해 임의로 구성한 서울여대 트렌드 txt
# 텍스트 데이터 로드
loader = TextLoader("./data/swuniv_trend.txt", encoding="utf-8")
documents = loader.load()

print(len(documents))

from langchain_core.documents import Document

import re

def split_restaurants(document):
    """
    Search campus-specific food culture and trends.

    This tool MUST be used first when the user asks about
    university food trends, menu ideas, or menu naming.

    It contains important information such as:
    - campus food preferences
    - campus culture
    - mascot cat names used for menu naming
    """

    pattern = r'([^\n]+\ncategory:.*?(?:\nmenu:.*?))(?=\n\n|$)'
    items = re.findall(pattern, document.page_content, re.DOTALL)

    restaurant_docs = []

    for item in items:

        lines = item.split("\n")
        name = lines[0].strip()

        category = ""
        menu = ""

        for line in lines:
            if "category:" in line:
                category = line.replace("category:", "").strip()
            if "menu:" in line:
                menu = line.replace("menu:", "").strip()

        # LangChain의 핵심 데이터 구조인 Document 객체로 포장. print(type())때 <class 'list'>등 으로 출력됨.
        # page_content는 검색 대상 내용, metadata는 부가 정보(필터링 등에 활용)
        restaurant_docs.append(
            Document(
                page_content=item.strip(),
                metadata={
                    "name": name,
                    "category": category,
                    "menu": menu,
                    "source": document.metadata["source"]
                }
            )
        )

    return restaurant_docs

# 문서 분할 실행
restaurant_documents = []
for doc in documents:
    restaurant_documents += split_restaurants(doc)

# 결과 출력
print(f"총 {len(restaurant_documents)}개의 메뉴 항목이 처리되었습니다.")

# 학교 마스코트 정보 추가 (고정된 규칙을 시스템에 주입하는 효과). txt에 정의해놨지만, 분할은 메뉴 파싱에 적합하게 해놔서 한번 더 작성.
school_info = Document(
    page_content="""
Seoul Women's University(서울여대) 학교 마스코트 고양이는 슈냥이다.
학교에는 여러 고양이가 살고 있으며 학생들에게 친숙한 존재이다.

고양이 이름:
우치, 우동, 랑, 삼색, 당고, 치즈

메뉴 이름을 만들 때는 반드시 위 고양이 이름만 사용해야 한다.
새로운 고양이 이름을 만들면 안 된다.
""",
    metadata={
        "type": "campus_info"
    }
)

restaurant_documents.append(school_info)

    
# Chroma Vectorstore를 사용하기 위한 준비
from langchain_chroma import Chroma
from langchain_ollama  import OllamaEmbeddings

# 임베딩 모델 및 Vector DB 설정
# 임베딩(Embedding): 텍스트를 컴퓨터가 의미를 이해할 수 있도록 '숫자(벡터)'로 변환하는 작업
# Ollama 홈페이지에서 각자 컴퓨터 사양에 맞게 다운받고, 모델 bge-m3도 다운받기!
embeddings_model = OllamaEmbeddings(model="bge-m3") 

# Chroma 인덱스 생성
univ_db = Chroma.from_documents(
    documents=restaurant_documents, 
    embedding=embeddings_model,   
    collection_name="univ_food",
    persist_directory="./campus_db", #저장경로 잘 지정하기
)

# Retriever: Vector DB에서 사용자 질문과 의미적으로 가장 유사한 문서를 '찾아오는(Retrieve)' 역할. 리트리버 강아지 생각하면 됨
univ_retriever = univ_db.as_retriever(
    #search_kwargs={'k': 4}, #default값은 유사도 기반으로 검색
    # MMR: 정확도와 다양성을 동시에 고려하여 검색 결과를 가져옵니다.
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 8
    }# 8개를 찾은 후, 가장 의미가 안 겹치는 4개를 반환
)

# 쿼리 테스트
query = "대학가에서 학생들이 좋아할 새로운 메뉴 이름을 만들고 싶습니다. 학교 마스코트 고양이 이름을 활용한 재미있는 메뉴 이름을 추천해주세요."
docs = univ_retriever.invoke(query)
print(f"검색 결과: {len(docs)}개")

#for doc in docs:
#    print(f"메뉴 이름: {doc.metadata['name']}")
#    print()

@tool
def myschooltrend(query: str):
    """
    Securely retrieve and access authorized restaurant information from the encrypted database.
    Use this tool only for seoul women's university queries to maintain data confidentiality.
    """
    docs = univ_retriever.invoke(query)

    return "\n\n".join([doc.page_content for doc in docs])

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


1
총 8개의 메뉴 항목이 처리되었습니다.
검색 결과: 4개


In [24]:
# =====================================================================
# [3단계] Agent 생성 및 실행 (두뇌 만들기)
# =====================================================================
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI

# ChatOpenAI 모델 초기화 (여기서 GPT가 Agent의 '사고력'을 담당)
llm = ChatOpenAI(model="gpt-4o-mini")

# Agent용 프롬프트 설계
# Agent가 도구를 어떤 '순서'로 써야 하는지, 어떤 '규칙(한국어 사용 등)'을 지켜야 하는지 시스템 프롬프트에 명확하게 지시하면 돼
agent_prompt = ChatPromptTemplate.from_messages([
    ("system", dedent("""
You are an AI assistant that recommends food trend around Seoul Women's University(서울여대) areas.

Your goal is to suggest promising food menus for campus businesses by analyzing:
1. Campus-specific trends
2. Current Korean search trends
3. General web information if needed

You have access to the following tools:

myschooltrend:
Use this tool to search campus-specific food trends stored in a local knowledge base.
This reflects what students commonly eat or prefer near the university.

naver_trend:
Use this tool to analyze how popular a food keyword is in recent Korean search trends.
This helps determine whether demand for a menu is increasing or decreasing.

search_web:
Use this tool when you need additional general information from the web.

Guidelines:
1. Always check campus trends first using myschooltrend.
2. Then analyze demand using naver_trend.
3. Campus mascot cat names should be used creatively when generating new menu names.
4. Use search_web only when additional context is needed.
5. When recommending a menu, explain why it fits the campus area.
6. Mention whether the trend demand is increasing or decreasing.
7. Provide clear, concise, and helpful explanations.
8. Maintain a friendly and conversational chatbot tone.

Your main objective is to recommend menus that could work well for students near universities.
                      
When naming new menu, use the school cat names. 
                      
Important rule:
All responses must be written in Korean.
Even if the tools or system messages are in English, the final answer must always be in Korean.
Think in English if needed, but always answer in Korean.                      
""")),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# Tool calling Agent 생성
from langchain.agents import AgentExecutor, create_tool_calling_agent
# 사용할 도구들을 리스트로 묶음
tools = [search_web, myschooltrend, naver_trend]
# Agent 생성: LLM, 도구, 프롬프트를 하나로 결합
agent = create_tool_calling_agent(llm, tools, agent_prompt)

# AgentExecutor: Agent가 목표를 달성할 때까지 도구를 반복해서 사용하도록 관리하는 '실행기'
# verbose=True로 설정하면 터미널에서 Agent의 생각 과정을 자세히 볼 수 있어 디버깅하기 좋음
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# AgentExecutor test
query = "요즘 대학가에서 잘 팔릴 메뉴 추천해줘"
agent_response = agent_executor.invoke({"input": query})

print(agent_response["output"])



> Entering new AgentExecutor chain...

Invoking: `myschooltrend` with `{'query': '서울여대'}`



Seoul Women's University(서울여대) 학교 마스코트 고양이는 슈냥이다.
학교에는 여러 고양이가 살고 있으며 학생들에게 친숙한 존재이다.

고양이 이름:
우치, 우동, 랑, 삼색, 당고, 치즈

메뉴 이름을 만들 때는 반드시 위 고양이 이름만 사용해야 한다.
새로운 고양이 이름을 만들면 안 된다.



Seoul Women's University(서울여대) 학교 마스코트 고양이는 슈냥이다.
학교에는 여러 고양이가 살고 있으며 학생들에게 친숙한 존재이다.

고양이 이름:
우치, 우동, 랑, 삼색, 당고, 치즈

메뉴 이름을 만들 때는 반드시 위 고양이 이름만 사용해야 한다.
새로운 고양이 이름을 만들면 안 된다.



Seoul Women's University(서울여대) 학교 마스코트 고양이는 슈냥이다.
학교에는 여러 고양이가 살고 있으며 학생들에게 친숙한 존재이다.

고양이 이름:
우치, 우동, 랑, 삼색, 당고, 치즈

학생들은 이 고양이들을 매우 좋아하며 학교의 상징적인 존재로 여긴다.
새로운 메뉴 이름을 만들 때 고양이 이름을 활용하면 재미있는 작명이 될 수 있다.


츄밥
category: 덮밥

더큰도시락
category: 도시락

버거ING
category: 햄버거

퀴즈노스
category: 샌드위치

공차
category: 버블티 음료

컴포즈커피
category: 카페, 커피



[Campus Restaurants - 50주년기념관 1층]

감탄떡볶이
category: 분식
menu: 떡볶이
Invoking: `naver_trend` with `{'keyword': '덮밥'}`


200
{"startDate":"2025-01-01","endDate":"2026-03-17","timeUnit":"month","results":[{"title":"덮밥","ke

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
#HumanMessage (사용자 메시지): 사람이 챗봇에게 입력한 질문이나 프롬프트를 담는 객체 (예: "요즘 대학가에서 잘 팔릴 메뉴 추천해줘")
#AIMessage (AI 메시지): 챗봇(LLM)이 생성한 답변을 담는 객체
#SystemMessage (시스템 메시지): AI에게 역할이나 규칙을 부여할 때 사용하는 객체 (예: "너는 서울여대 주변 식당 트렌드 전문가야.")

import gradio as gr
from typing import List, Tuple

def answer_invoke(message: str, history: List[Tuple[str, str]]) -> str:
    try:
        # 채팅 기록을 AI에게 전달할 수 있는 형식으로 변환
        chat_history = []
        for human, ai in history:
            chat_history.append(HumanMessage(content=human))
            chat_history.append(AIMessage(content=ai))
        
        # agent_executor를 사용하여 응답 생성
        response = agent_executor.invoke({
            "input": message,
            "chat_history": chat_history[-2:]    # 최근 2개의 메시지 기록만을 활용 
        })
        
        # agent_executor의 응답에서 최종 답변 추출
        return response['output']
    except Exception as e:
        # 오류 발생 시 사용자에게 알리고 로그 기록
        print(f"Error occurred: {str(e)}")
        return "죄송합니다. 응답을 생성하는 동안 오류가 발생했습니다. 다시 시도해 주세요."

# 예제 질문 정의
example_questions = [
"서울여대 대학가에서 가게를 운영 중인데 학생들이 좋아할 메뉴를 추가하고 싶습니다. 메뉴 이름을 작명해 주세요",

"서울여대 대학가에서 두쫀쿠를 창업하고 싶습니다. 학교 주변 음식 트렌드와 최근 검색 트렌드를 분석해서 앞으로 유행할 메뉴를 알려주세요.",

"서울여대 대학생들이 좋아할 만한 다음 유행 음식 메뉴를 추천해주세요. 학교 주변 트렌드와 네이버 검색 트렌드를 기반으로 분석해주세요.",


]

# Gradio 인터페이스 생성
demo = gr.ChatInterface(
    fn=answer_invoke,
    title="두.쫀.쿠 대학가 창업 및 메뉴 추가 AI 어시스턴트",
    description="창업 추천,메뉴 추천, 음식 관련 질문에 답변해 드립니다.",
    examples=example_questions,
    theme=gr.themes.Soft()
)

# 데모 실행
#demo.launch()
demo.launch(share=True) #share=True시 public URL 제공(단 로컬 서버는 켜놔야하고 72시간만 유효)

Running on local URL:  http://127.0.0.1:7860
Running on public URL: https://6f568afae528b0faa2.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)




> Entering new AgentExecutor chain...

Invoking: `myschooltrend` with `{'query': '서울여대 학생들이 선호하는 음식'}`



Seoul Women's University(서울여대) 학교 마스코트 고양이는 슈냥이다.
학교에는 여러 고양이가 살고 있으며 학생들에게 친숙한 존재이다.

고양이 이름:
우치, 우동, 랑, 삼색, 당고, 치즈

학생들은 이 고양이들을 매우 좋아하며 학교의 상징적인 존재로 여긴다.
새로운 메뉴 이름을 만들 때 고양이 이름을 활용하면 재미있는 작명이 될 수 있다.



Seoul Women's University(서울여대) 학교 마스코트 고양이는 슈냥이다.
학교에는 여러 고양이가 살고 있으며 학생들에게 친숙한 존재이다.

고양이 이름:
우치, 우동, 랑, 삼색, 당고, 치즈

메뉴 이름을 만들 때는 반드시 위 고양이 이름만 사용해야 한다.
새로운 고양이 이름을 만들면 안 된다.



Seoul Women's University(서울여대) 학교 마스코트 고양이는 슈냥이다.
학교에는 여러 고양이가 살고 있으며 학생들에게 친숙한 존재이다.

고양이 이름:
우치, 우동, 랑, 삼색, 당고, 치즈

메뉴 이름을 만들 때는 반드시 위 고양이 이름만 사용해야 한다.
새로운 고양이 이름을 만들면 안 된다.


츄밥
category: 덮밥

더큰도시락
category: 도시락

버거ING
category: 햄버거

퀴즈노스
category: 샌드위치

공차
category: 버블티 음료

컴포즈커피
category: 카페, 커피



[Campus Restaurants - 50주년기념관 1층]

감탄떡볶이
category: 분식
menu: 떡볶이
Invoking: `naver_trend` with `{'keyword': '덮밥'}`


200
{"startDate":"2025-01-01","endDate":"2026-03-17","timeUnit":"month","results":[{"ti

In [28]:
demo.close()

Closing server running on port: 7860
